# `app/views/overview.py`

Overview page.

In [ ]:
# Setup: work from the project root so `src` / `app` imports and relative data paths resolve,
# wherever Jupyter was started.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import streamlit as st

from app.views.common import banner, footer, require_results
from src.webapp.results_io import MODEL_ORDER

In [ ]:
def render() -> None:
    st.title("Heart Disease Risk Modelling")
    st.subheader("Phase 1: six machine-learning classifiers on the Cleveland heart disease data")
    banner()
    st.markdown(
        "This app accompanies an academic study that compares six classifiers for angiographically defined heart disease "
        "on the Cleveland subset of the UCI Heart Disease data (303 patients): a Bayesian network, Naive Bayes, "
        "Logistic Regression, an SVM, a Random Forest and Gradient Boosting. Use the pages on the left to explore the "
        "data, inspect the evaluation results, try the models on a hypothetical profile, and read how the models were "
        "built and where they fall short.")
    res = require_results()
    st.markdown("#### Headline results (nested cross-validation, 25 identical outer folds per model)")
    ci = res["ci"]
    for start in range(0, len(MODEL_ORDER), 3):
        for col, model in zip(st.columns(3), MODEL_ORDER[start:start + 3]):
            r = ci[ci.model == model].set_index("metric")
            with col:
                st.metric(f"{model}: ROC AUC", f"{r.loc['roc_auc', 'estimate']:.3f}",
                          help=f"95% bootstrap interval [{r.loc['roc_auc', 'ci_low']:.3f}, {r.loc['roc_auc', 'ci_high']:.3f}]")
                st.caption(f"95% CI [{r.loc['roc_auc', 'ci_low']:.3f}, {r.loc['roc_auc', 'ci_high']:.3f}] · "
                           f"accuracy {r.loc['accuracy', 'estimate']:.3f}")
    diff = res["diff"]
    auc = diff[diff.metric == "roc_auc"]
    separated = auc[~auc.ci_includes_zero].comparison.tolist()
    ranking = ("no paired ROC AUC difference is distinguishable from zero, so the models cannot be told apart."
               if not separated else
               f"{len(auc) - len(separated)} of {len(auc)} paired ROC AUC differences include zero; only "
               + ", ".join(separated) + " is distinguishable (see Model results).")
    st.markdown(
        f"- **Ranking:** {ranking}\n"
        "- **Limits:** one centre, 303 patients, internal validation only. These numbers do not show clinical validity.")
    footer()